# Unidad 5. Almacenamiento y acceso a datos a gran escala

En este cuaderno vas a cruzar la base de datos interna de términos de la **Red Iberia de Traducción Jurídica** con un glosario externo de referencia, para el encargo de **Términos & Trama**.

## Qué vas a practicar

1. Conectar con una base de datos SQLite y consultarla con SQL desde Pandas.
2. Cargar una fuente de datos externa (glosario en CSV).
3. Normalizar y cruzar ambas fuentes.
4. Calcular un informe de cobertura terminológica.
5. Exportar el resultado a CSV y de vuelta a la base de datos.

## 1. Preparación del entorno

In [ ]:
import sqlite3

# Si falta pandas, activa el entorno virtual .venv de la asignatura e instala:
# python -m pip install -r 03_unidades/U5_almacenamiento_acceso_datos_externas/requirements_U5.txt
import pandas as pd

## 2. Consultar la base de datos interna

In [ ]:
conexion = sqlite3.connect("materiales/terminos_internos.db")

df_internos = pd.read_sql_query("SELECT * FROM terminos_internos", conexion)
df_internos.head()

## 3. Cargar el glosario externo

In [ ]:
# En un caso real, esta misma función podría apuntar a una URL:
# df_externo = pd.read_csv("https://ejemplo.org/glosario_publico.csv")
df_externo = pd.read_csv("materiales/glosario_externo.csv")
df_externo.head()

## 4. Normalizar y cruzar ambas fuentes

In [ ]:
df_internos["termino_normalizado"] = df_internos["termino"].str.strip().str.lower()
df_externo["termino_normalizado"] = df_externo["termino_es"].str.strip().str.lower()

df_cruce = df_internos.merge(
    df_externo[["termino_normalizado", "equivalente_en"]],
    on="termino_normalizado", how="left"
)
df_cruce["tiene_equivalente"] = df_cruce["equivalente_en"].notna()

df_cruce.head()

## 5. Informe de cobertura

In [ ]:
resumen = df_cruce["tiene_equivalente"].value_counts()
resumen

## 6. Exportar el resultado

In [ ]:
df_cruce.to_csv("U5_informe_cobertura.csv", index=False, encoding="utf-8")
df_cruce.to_sql("informe_cobertura", conexion, if_exists="replace", index=False)

conexion.close()

## 7. Diagnóstico

Responde aquí, en pocas frases: ¿qué proporción de los términos internos tiene ya equivalente en el glosario externo? ¿Qué le recomendarías a la Red Iberia de Traducción Jurídica: encargar una revisión terminológica completa, o centrarla solo en los términos sin equivalente? Justifica con las cifras obtenidas.

*Escribe tu respuesta aquí.*

## 8. Bloque final opcional: un diccionario bilingüe real y grande

Todo lo que has hecho en este cuaderno funciona igual si, en vez del glosario sintético de 22 términos, el glosario externo tiene más de cien mil pares de palabras. Vamos a comprobarlo con un diccionario bilingüe inglés-español real: el diccionario de MUSE (Meta AI Research), con 112.580 pares de palabras, de uso libre para fines no comerciales (licencia CC BY-NC 4.0).

Este bloque es **opcional y no se evalúa**. El archivo pesa solo unos 2 MB, así que no hace falta descargarlo antes de la sesión: se carga directamente desde la URL en la propia celda.

### Cargar el diccionario real

In [ ]:
diccionario_real = pd.read_csv(
    "https://dl.fbaipublicfiles.com/arrival/dictionaries/en-es.txt",
    sep=" ",
    names=["termino_en", "termino_es"],
)
diccionario_real.shape

### Guardarlo como una tabla nueva en una base de datos (sin tocar la original)

In [ ]:
con_real = sqlite3.connect("materiales/diccionario_real.db")
diccionario_real.to_sql("diccionario_real", con_real, if_exists="replace", index=False)
con_real.close()

### Comparar la escala con la base de datos de la unidad

In [ ]:
comparacion = pd.DataFrame({
    "tabla": ["terminos_internos (unidad)", "diccionario_real (MUSE)"],
    "filas": [len(df_internos), len(diccionario_real)],
})
comparacion

### ¿Cuánta cobertura léxica tienen los términos internos, palabra a palabra?

En vez de buscar cada término completo tal cual (una frase jurídica compuesta casi nunca coincidirá con una entrada de un diccionario general de palabras sueltas), vamos a comprobar cuántas de las palabras que forman cada término aparecen en el diccionario real.

In [ ]:
def cobertura_lexica(termino):
    palabras = termino.lower().split()
    cubiertas = [p for p in palabras if p in terminos_es_dic]
    return len(cubiertas) / len(palabras)

terminos_es_dic = set(diccionario_real["termino_es"].str.lower())
df_internos["cobertura_lexica"] = df_internos["termino"].apply(cobertura_lexica)
df_internos[["termino", "cobertura_lexica"]].drop_duplicates().sort_values("cobertura_lexica")

In [ ]:
cobertura_por_termino = df_internos[["termino", "cobertura_lexica"]].drop_duplicates()
cobertura_media = cobertura_por_termino["cobertura_lexica"].mean()
terminos_totalmente_cubiertos = (cobertura_por_termino["cobertura_lexica"] == 1.0).sum()
cobertura_media, terminos_totalmente_cubiertos

### Reflexión final (bloque opcional)

Mira los términos con menor cobertura léxica (los que aparecen arriba del todo en la tabla ordenada). Responde en pocas frases: ¿qué tienen en común esas palabras concretas que no aparecen en el diccionario general? ¿Tiene sentido que un diccionario bilingüe de uso general no las recoja?

*Escribe tu respuesta aquí.*